# Medical Triage Data Preprocessing Pipeline

## Abstract
This notebook implements a comprehensive preprocessing pipeline for medical triage data, designed to prepare structured datasets for machine learning applications in emergency department decision support systems. The pipeline consists of two main components: 
- (1) conversion of clinical guideline documents from DOCX to structured JSON format, and 
- (2) systematic processing of medical case notes from PDF through multiple intermediate formats to create clean, structured datasets suitable for Retrieval-Augmented Generation (RAG) systems.

## Overview
The preprocessing pipeline addresses the challenge of converting heterogeneous medical documentation into standardized, machine-readable formats while preserving clinical relevance and maintaining data quality throughout the transformation process.

In [ ]:
import os
import time
import json
import re
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Union
from difflib import SequenceMatcher
import pandas as pd
from docx import Document
from docx.table import _Cell, Table
from docx.text.paragraph import Paragraph
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from unstructured.partition.doc import partition_doc
from unstructured.documents.elements import (
    Title,
    NarrativeText,
    ListItem,
    Text,
    Table as UnstructuredTable,
)

## 1. Triage Guideline Processing Module

### 1.1 Objective
This module converts clinical triage guidelines from DOCX format into structured JSON documents, enabling systematic retrieval and analysis of clinical decision-making criteria.

### 1.2 Methodology
The processing workflow consists of two sequential stages:
- **Document Conversion**: DOCX files are converted to Markdown format while preserving document structure and table formatting
- **Content Segmentation**: Markdown content is systematically divided into discrete sections based on heading hierarchy for optimized retrieval operations

### 1.3 Implementation

#### 1.1 DOCX to Markdown Conversion

**Purpose**: Converts clinical guideline documents from proprietary DOCX format to standardized Markdown while preserving structural elements including headings, tables, and nested lists.

**Key Features**:
- Automated heading level detection and preservation
- Table structure maintenance with proper Markdown formatting
- Bullet point hierarchy preservation for clinical decision trees
- Text cleaning and normalization protocols

In [ ]:
@dataclass
class HeadingInfo:
    """Store heading information including position"""

    level: int
    position: int
    text: str

def get_heading_positions(doc: Document) -> List[HeadingInfo]:
    """Create a list of headings with their positions using python-docx."""
    headings = []

    for pos, para in enumerate(doc.paragraphs):
        text = clean_text(para.text)
        if text and "Heading" in para.style.name:
            try:
                level = int(para.style.name.replace("Heading ", ""))
                heading = HeadingInfo(level=level, position=pos, text=text)
                headings.append(heading)
            except ValueError:
                continue
    return headings

def get_paragraph_level(paragraph: Paragraph) -> int:
    """Determine the nesting level of a paragraph."""
    level = 0
    if hasattr(paragraph._p, "pPr") and paragraph._p.pPr is not None:
        numPr = paragraph._p.pPr.xpath(".//w:numPr")
        if numPr:
            ilvl = numPr[0].xpath(".//w:ilvl")
            if ilvl:
                try:
                    level = int(ilvl[0].get(qn("w:val"))) + 1
                except (ValueError, TypeError):
                    pass
    return level


def clean_text(text: str) -> str:
    """Clean up text by removing extra whitespace and normalizing newlines."""
    text = " ".join(text.split()).strip()
    if text.startswith("{#") and text.endswith(".unnumbered}"):
        text = text.replace("{#", "").replace(".unnumbered}", "")
    return text


def format_cell_content(cell: _Cell) -> str:
    """Format cell content with bullet points in a RAG-friendly format."""
    content_parts = []

    for paragraph in cell.paragraphs:
        text = paragraph.text.strip()
        if not text:
            continue

        level = get_paragraph_level(paragraph)
        if level > 0:
            indent = "  " * (level - 1)
            bullet = "•" if level == 1 else "○"
            content_parts.append(f"{indent}{bullet} {text}")
        else:
            content_parts.append(text)

    return (" ".join(content_parts)).replace("|", "\\|")


def create_markdown_table(table: Table) -> str:
    """Convert a docx table to markdown format optimized for RAG."""
    if not table.rows:
        return ""

    md_lines = []

    # Header row
    header_cells = [format_cell_content(cell) for cell in table.rows[0].cells]
    header = "| " + " | ".join(header_cells) + " |"
    md_lines.append(header)

    # Separator row
    separator = "|" + "|".join("-" * len(cell) for cell in header_cells) + "|"
    md_lines.append(separator)

    # Data rows
    for row in table.rows[1:]:
        cells = [format_cell_content(cell) for cell in row.cells]
        while len(cells) < len(header_cells):
            cells.append("")
        row_md = "| " + " | ".join(cells) + " |"
        md_lines.append(row_md)

    return "\n".join(md_lines)


def convert_doc_to_markdown(input_file: str) -> str:
    """
    Convert DOCX to markdown with proper heading levels and formatted tables.
    """
    if not os.path.exists(input_file):
        raise FileNotFoundError(f"Input file not found: {input_file}")

    # Get document for heading and table processing
    doc = Document(input_file)

    # Get heading information
    headings = get_heading_positions(doc)
    heading_map = {h.text: h.level for h in headings}

    # Process document with unstructured for general content
    elements = partition_doc(filename=input_file)

    markdown_content = []
    current_pos = 0

    for element in elements:
        text = clean_text(element.text)
        if not text:
            continue

        if isinstance(element, UnstructuredTable):
            # When encounter a table, process it using python-docx
            table_index = len(
                [
                    e
                    for e in elements[: elements.index(element)]
                    if isinstance(e, UnstructuredTable)
                ]
            )
            if table_index < len(doc.tables):
                table_md = create_markdown_table(doc.tables[table_index])
                markdown_content.append(f"\n{table_md}\n")
        else:
            # Handle regular content
            if text in heading_map:
                level = heading_map[text]
                markdown_content.append(f"\n{'#' * level} {text}\n")
            elif isinstance(element, ListItem):
                markdown_content.append(f"- {text}\n")
            else:
                markdown_content.append(f"{text}\n")

        current_pos += 1

    return "\n".join(markdown_content)


def process_and_save_document(input_file: str, output_file: str) -> str:
    """
    Process document and save the result.
    """
    try:
        markdown_content = convert_doc_to_markdown(input_file)

        # Save if output file specified
        if output_file:
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(markdown_content)
            print(f"Markdown saved to: {output_file}")

        return markdown_content

    except Exception as e:
        print(f"Error processing document: {str(e)}")
        raise

#### 1.2 Content Segmentation

**Purpose**: Segments processed Markdown content into discrete, retrievable units based on clinical topic boundaries defined by heading structure.

**Segmentation Strategy**: Level 2 headers (##) serve as primary segmentation boundaries, ensuring each section contains coherent clinical guidance suitable for retrieval operations.

In [ ]:
def split_triage_guidelines(file_path: str) -> List[Dict]:
    """
    Split a markdown file containing triage guidelines into sections based on level 2 headers (##)

    Args:
        file_path (str): Path to the markdown file

    Returns:
        List[Dict]: List of document chunks with metadata
    """
    # Define headers to split on - only want level 2 headers (##)
    headers_to_split_on = [
        ("##", "section_title"),
    ]

    # Initialize markdown splitter
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on,
        strip_headers=False,
    )

    try:
        # Read the markdown file
        with open(file_path, "r", encoding="utf-8") as file:
            content = file.read()

        # Split the document
        splits = markdown_splitter.split_text(content)

        print(f"Successfully processed triage guidelines")
        print(f"Total sections found: {len(splits)}")

        return splits

    except Exception as e:
        print(f"Error processing file: {str(e)}")
        return []


def save_sections_to_json(splits: List[Dict], output_file: str):
    """
    Save the splits to a JSON file

    Args:
        splits (List[Dict]): List of document splits
        output_file (str): Path to output JSON file
    """
    sections_data = []

    for split in splits:
        section_data = {
            "section_title": split.metadata.get("section_title", "Untitled Section"),
            "content": split.page_content,
        }
        sections_data.append(section_data)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(sections_data, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(sections_data)} sections to {output_file}")


def load_sections_from_json(input_file: str) -> List[Dict[str, Any]]:
    """
    Load sections from a JSON file

    Args:
        input_file (str): Path to input JSON file

    Returns:
        List[Dict[str, Any]]: List of section data
    """
    with open(input_file, "r", encoding="utf-8") as f:
        sections = json.load(f)

    print(f"Loaded {len(sections)} sections from {input_file}")
    return sections

### 1.3 Execution and Results
**Guideline Processing Implementation**

In [ ]:
# Define file paths
input_docx = "../data/guideline/triage_guideline_sections.docx"
markdown_output = "../data/guideline/triage_guideline_sections.md"
sections_json = "../data/guideline/triage_guideline_sections.json"

In [ ]:
# Step 1: Convert DOCX to Markdown
print("\n=== Step 1: Converting DOCX to Markdown ===")
try:
    markdown_content = process_and_save_document(input_docx, markdown_output)
    print("Conversion complete!")
except Exception as e:
    print(f"Error in Step 1: {str(e)}")

In [ ]:
# Step 2: Split Markdown into sections
print("\n=== Step 2: Splitting Markdown into sections ===")
sections = split_triage_guidelines(markdown_output)
save_sections_to_json(sections, sections_json)

print("\\nProcess complete! Outputs saved to:")
print(f"- {markdown_output}")
print(f"- {sections_json}")

## 2. Medical Case Notes Processing Pipeline

### 2.1 Objective
This module implements a multi-stage processing pipeline to transform medical case documentation from unstructured PDF format into structured JSON records suitable for machine learning applications and clinical decision support systems.

### 2.2 Pipeline Architecture
The processing pipeline consists of four sequential stages designed to ensure data quality and clinical relevance:

1. **Document Extraction**: PDF case notes → Anonymized text files
2. **Data Structuring**: Text files → Structured CSV format with clinical field extraction
3. **Quality Assurance**: Manual review checkpoint for data validation
4. **Format Standardization**: CSV data → Structured JSON with hierarchical organization
5. **Dataset Partitioning (optional)**: Random stratified sampling for validation and test set creation (10% each). ONLY enabled during pipeline experiments

### 2.3 Clinical Data Fields
The pipeline extracts and structures the following clinical information categories:
- **Demographics**: Patient characteristics and communication preferences
- **Clinical Presentation**: Chief complaints and arrival conditions
- **Vital Signs**: Physiological measurements and monitoring data
- **Case Disposition**: Triage decisions, diagnoses, and discharge planning

#### 2.1 PDF to Text Conversion
**Purpose**: Extracts textual content from PDF case notes while implementing anonymization protocols to protect patient confidentiality.

In [ ]:
import pdfplumber
import re
import os
from pathlib import Path

In [ ]:
def anonymize_medical_note(text):
    """
    Remove confidential information from medical notes while preserving content structure.
    Keeps staff names intact.

    Args:
        text (str): Raw text from the medical note
    Returns:
        str: Partially anonymized text with staff names preserved
    """
    # Split text into lines for processing
    lines = text.split("\n")
    cleaned_lines = []

    # Flag to track if we're in the header section
    in_header = True

    # Footer patterns
    footer_patterns = [
        r"This A&E Clinical Documentation Form is intended for.*",
        r"此乃急症室臨床記錄.*",
        r"For professional communication only\.",
        r"只供醫護人員參考。",
        r"Generated on.*Page \d+ of \d+",
        r"-- End --",
    ]

    for line in lines:
        # Skip empty lines
        if not line.strip():
            continue

        # Check if we've reached the end of the header
        if in_header and line.strip().startswith("Sex:"):
            in_header = False

        # Skip if we're still in the header
        if in_header:
            continue

        # Skip if line matches any footer pattern
        if any(re.match(pattern, line.strip()) for pattern in footer_patterns):
            continue

        # No longer replacing staff names - they will be preserved in the output
        cleaned_lines.append(line)

    # Join lines back together
    cleaned_text = "\n".join(cleaned_lines)

    # Remove any multiple consecutive newlines
    cleaned_text = re.sub(r"\n{3,}", "\n\n", cleaned_text)

    return cleaned_text.strip()


def remove_repeating_headers(text):
    """
    Remove repeating header sections that are preceded by a blank line.
    Keep only the first occurrence of the header.

    Args:
        text (str): Text with potentially repeating headers
    Returns:
        str: Text with only the first header occurrence
    """
    # Extract the first occurrence of the header (first two lines)
    lines = text.split("\n")
    if len(lines) < 2:
        return text

    # Get the first two lines as the header pattern
    header_line1 = lines[0]
    header_line2 = lines[1]

    # Create a pattern to match repeating headers preceded by a blank line
    # Pattern explanation:
    # (\n\s*\n) - Capture group for the blank line (to remove it as well)
    # followed by exact matches for the first two header lines
    header_pattern = (
        r"(\n\s*\n)" + re.escape(header_line1) + r"\n" + re.escape(header_line2)
    )

    # Replace all occurrences with an empty string (removing both blank line and header)
    cleaned_text = re.sub(header_pattern, "", text)

    return cleaned_text


def batch_process_pdfs(input_dir, output_dir):
    """
    Process all PDF files in the input directory and its subdirectories, maintaining
    the same folder structure in the output directory.

    Args:
        input_dir (str): Path to directory containing PDF files
        output_dir (str): Path to directory where anonymized files will be saved
    """
    # Convert to Path objects for easier path manipulation
    input_path = Path(input_dir)
    output_path = Path(output_dir)

    # Walk through all directories and files
    for root, dirs, files in os.walk(input_path):
        # Get the relative path from input directory
        rel_path = Path(root).relative_to(input_path)

        # Create corresponding output directory
        current_output_dir = output_path / rel_path
        os.makedirs(current_output_dir, exist_ok=True)

        # Process PDF files in current directory
        pdf_files = [f for f in files if f.lower().endswith(".pdf")]

        for pdf_file in pdf_files:
            try:
                # Construct full input and output paths
                input_file_path = Path(root) / pdf_file
                output_filename = f"{pdf_file[:-4]}_anonymized.txt"
                output_file_path = current_output_dir / output_filename

                # Process the PDF
                with pdfplumber.open(input_file_path) as pdf:
                    # Extract and anonymize text from all pages
                    all_cleaned_text = []
                    for page in pdf.pages:
                        text = page.extract_text()
                        if text:  # Check if text extraction was successful
                            cleaned_text = anonymize_medical_note(text)
                            all_cleaned_text.append(cleaned_text)

                    # Join all pages together
                    final_text = "\n\n".join(all_cleaned_text)

                    # Remove repeating headers from the final text
                    final_text = remove_repeating_headers(final_text)

                    # Save to output file
                    with open(output_file_path, "w", encoding="utf-8") as f:
                        f.write(final_text)

                print(f"Successfully processed: {input_file_path}")

            except Exception as e:
                print(f"Error processing {input_file_path}: {str(e)}")

    print(f"\nProcessing complete. Files saved to: {output_dir}")

In [ ]:
batch_process_pdfs(input_dir=r"..\data\past_case\pdf", output_dir=r"..\data\past_case\txt")

#### 2.2 Text to CSV Extraction
**Purpose**: Applies natural language processing techniques to extract structured clinical data fields from free-text case notes, organizing information into tabular format for systematic analysis.

In [ ]:
import pandas as pd
import re
from pathlib import Path
import os
import re


In [ ]:
def extract_medical_info(text):
    """
    Extract relevant medical information from anonymized text and return as a dictionary
    """
    info = {}

    # Extract key sections first for better organization
    def extract_section(section_name, text):
        section_patterns = {
            "triage": r"Triage\n(.*?)(?:Consultation|Investigation)",
            "vital_signs": r"Vital sign and observations(.*?)(?:Consultation|Investigation)",
            "consultation": r"Consultation\n(.*?)(?:Investigation|Reassessment|Re-Assess|Discharge)",
            "allergy": r"Allergy / Alert Information\n(.*?)(?=\nDiagnosis|\nRecent COVID-19 Information|\nTriage)",
        }

        pattern = section_patterns.get(section_name.lower())
        if not pattern:
            return None

        match = re.search(pattern, text, re.DOTALL)
        return match.group(1) if match else None

    # Extract main sections
    triage_section = extract_section("triage", text)
    vital_section = extract_section("vital_signs", text)
    consultation_section = extract_section("consultation", text)
    allergy_section = extract_section("allergy", text)

    # Basic Demographics
    sex_match = re.search(r"Sex:\s*([MF])", text)
    info["Sex"] = sex_match.group(1) if sex_match else None

    age_match = re.search(r"Age:\s*(\d+)(m)?", text)
    if age_match:
        age_value = age_match.group(1)
        # group(2) could be None if the optional group didn't match
        # Using the or operator to handle this case safely
        age_unit = age_match.group(2) or ""

        # If the unit is "m", it's months, otherwise it's years
        if age_unit == "m":
            info["Age"] = f"{age_value} months"  # Age in months
        else:
            info["Age"] = f"{age_value} years"  # Age in years (no suffix)
    else:
        info["Age"] = None

    # Triage Nurse
    triage_nurse_match = re.search(
        r"Triage time:.*?by\s+([A-Z]+(?:,\s*[A-Z\s]+)+)", triage_section
    )
    info["Triage Nurse"] = (
        triage_nurse_match.group(1).strip() if triage_nurse_match else None
    )

    # Case Doctor
    case_doctor_match = re.search(
        r"Case doctor:\s*([^:\n]+?)(?:\n|$)", consultation_section
    )
    info["Case Doctor"] = (
        case_doctor_match.group(1).strip() if case_doctor_match else None
    )

    # Allergy and Alert
    if allergy_section:
        # Join lines first to handle text split across multiple lines
        normalized_section = re.sub(r"\n\s*", " ", allergy_section)

        # Define the section boundaries based on the known order: Allergy → ADR → Alert
        sections = {
            "Allergy": {"start": "Allergy:", "end": ["ADR:", "Alert:"]},
            "ADR": {"start": "ADR:", "end": ["Alert:"]},
            "Alert": {"start": "Alert:", "end": []},  # No boundary for the last section
        }

        for item_type, boundaries in sections.items():
            start_pattern = re.escape(boundaries["start"])

            # Find the start position of this section
            start_match = re.search(start_pattern, normalized_section)
            if not start_match:
                info[item_type] = None
                continue

            start_pos = start_match.start()

            # Find the end position (start of the next section)
            end_pos = len(normalized_section)
            for end_marker in boundaries["end"]:
                end_match = re.search(re.escape(end_marker), normalized_section)
                if end_match and end_match.start() > start_pos:
                    end_pos = min(end_pos, end_match.start())

            # Extract the section text
            section_text = normalized_section[start_pos:end_pos].strip()

            # Remove the section heading
            section_content = section_text.replace(boundaries["start"], "").strip()

            # Split by semicolon and clean up
            if section_content:
                items = re.split(r";\s*", section_content)
                cleaned_items = [item.strip() for item in items if item.strip()]
                info[item_type] = "; ".join(cleaned_items)
            else:
                info[item_type] = None
    else:
        info["Allergy"] = None
        info["ADR"] = None
        info["Alert"] = None

    # Diagnosis
    diagnosis_match = re.search(
        r"Diagnosis\n\s*(.+?)(?=\n\(\? : Provisional\)|\nTriage|$)",
        text,
        re.DOTALL,
    )
    if diagnosis_match:
        diagnosis_text = diagnosis_match.group(1).strip()
        # Remove "Principal: " prefix if present
        diagnosis_text = re.sub(r"^Principal:\s*", "", diagnosis_text)
        info["Diagnosis"] = diagnosis_text
    else:
        info["Diagnosis"] = None

    # Triage Info
    triage_cat_match = re.search(r"Triage category:\s*(\d)", triage_section)
    info["Triage Category"] = triage_cat_match.group(1) if triage_cat_match else None

    # Ambulatory Status and Fall Risk
    amb_status_match = re.search(
        r"Amb status:\s*(.+?)(?=Risk of fall:|$)", triage_section
    )
    info["Ambulatory status"] = (
        amb_status_match.group(1).strip() if amb_status_match else None
    )

    fall_risk_match = re.search(r"Risk of fall:\s*(.+?)(?=\n|$)", triage_section)
    info["Risk of fall"] = fall_risk_match.group(1).strip() if fall_risk_match else None

    # Communication and Informant
    informant_match = re.search(
        r"Informant:\s*(.+?)(?=Communication|$)", triage_section
    )
    info["Informant"] = informant_match.group(1).strip() if informant_match else None

    comm_match = re.search(r"Communication:\s*(.+?)(?=\n|$)", triage_section)
    info["Communication"] = comm_match.group(1).strip() if comm_match else None

    # Chief Complaint and Condition
    chief_match = re.search(
        r"Chief complaint:\s*(.+?)(?=\nCondition|\nPast health:|\nTOCC:|\nReferral:|\nVital sign|\nTriage intervention|\nConsultation|$)",
        triage_section,
        re.DOTALL,
    )
    info["Chief complaint"] = chief_match.group(1).strip() if chief_match else None

    # Condition on arrival - capture multiple lines until Past health:
    condition_match = re.search(
        r"Condition on arrival:\s*(.*?)(?=\nPast health:|\nTOCC:|\nReferral:|\nVital sign|\nTriage intervention|\nConsultation|$)",
        triage_section,
        re.DOTALL,
    )
    info["Condition on arrival"] = (
        condition_match.group(1).strip() if condition_match else None
    )

    # Past Health and TOCC
    past_health_match = re.search(
        r"Past health:\s*(.+?)(?=\nTOCC:|\nReferral:|\nVital sign|\nTriage intervention|\nConsultation|$)",
        triage_section,
        re.DOTALL,
    )
    info["Past health"] = (
        past_health_match.group(1).strip() if past_health_match else None
    )

    tocc_match = re.search(r"TOCC:\s*(.+?)(?=\n|$)", triage_section)
    info["TOCC"] = tocc_match.group(1).strip() if tocc_match else None

    # Extract Referral
    referral_match = re.search(r"Referral:\s*(.+?)(?=\n|$)", triage_section)
    info["Referral"] = referral_match.group(1).strip() if referral_match else None

    # Use vital_section when available, fall back to original triage_section
    section_for_vitals = vital_section if vital_section else triage_section

    # Vital Signs
    gcs_match = re.search(
        r"GCS:\s*(E:\s*\d+,\s*V:\s*\d+,\s*M:\s*\d+,\s*Score:\s*\d+/\d+)",
        section_for_vitals,
    )
    info["GCS"] = gcs_match.group(1) if gcs_match else None

    # Enhanced BP extraction
    def extract_bp_readings(section_text):
        if not section_text:
            return None

        # First BP reading pattern - modified to better capture position
        first_bp_pattern = r"BP:\s*1st\s*(\d+)\s*/\s*(\d+)\s*mmHg,\s*([^,\n]+?)(?:\s*,\s*--\s*PR/AR|\s*,\s*--)"

        # Second BP reading pattern - modified to better capture position
        second_bp_pattern = (
            r"2nd\s*(\d+)\s*/\s*(\d+)\s*mmHg,\s*([^,\n]+?)(?:\s*,\s*--\s*2nd|\s*,\s*--)"
        )

        readings = []

        # Extract first reading
        first_match = re.search(first_bp_pattern, section_text)
        if first_match:
            systolic, diastolic, position = first_match.groups()
            position = position.strip()
            reading = f"1st: {systolic}/{diastolic} mmHg"
            if position and position != "--":
                reading += f" ({position})"
            readings.append(reading)

        # Extract second reading
        second_match = re.search(second_bp_pattern, section_text)
        if second_match:
            systolic, diastolic, position = second_match.groups()
            position = position.strip()
            reading = f"2nd: {systolic}/{diastolic} mmHg"
            if position and position != "--":
                reading += f" ({position})"
            readings.append(reading)

        return "; ".join(readings) if readings else None

    info["BP"] = extract_bp_readings(section_for_vitals)

    pr_match = re.search(r"PR/AR:\s*1st\s*(\d+)\s*/min", section_for_vitals)
    info["PR/AR"] = pr_match.group(1) if pr_match else None

    temp_match = re.search(r"Temp:\s*([\d.]+)°C", section_for_vitals)
    info["Temp"] = temp_match.group(1) if temp_match else None

    # Extract SpO2

    spo2_pattern = r"SpO\s*:?\s*(\d+)%\s*on\s*([^R]+?)(?=RR:|$)"
    spo2_match = re.search(spo2_pattern, section_for_vitals)
    if spo2_match:
        percentage, condition = spo2_match.groups()
        condition = condition.strip().rstrip(",").strip()

        # Fix for the "Non-rebreathing" without "mask" issue
        if "Non-rebreathing" in condition and "mask" not in condition.lower():
            condition = condition.replace("Non-rebreathing", "Non-rebreathing mask")

        info["SpO2"] = f"{percentage}% on {condition}"
    else:
        info["SpO2"] = None

    # Extract RR
    rr_pattern = r"RR:\s*((?:\d+(?:\s*/min)?)|(?:--))"
    rr_match = re.search(rr_pattern, section_for_vitals)
    if rr_match:
        rr_value = rr_match.group(1)
        if rr_value != "--":
            number_match = re.search(r"\d+", rr_value)
            if number_match:
                rr_value = number_match.group()
        info["RR"] = rr_value
    else:
        info["RR"] = None

    # Extract PFR
    pfr_pattern = r"PFR:\s*([^L\n]+?)(?=Limbs:|$)"
    pfr_match = re.search(pfr_pattern, section_for_vitals)
    info["PFR"] = pfr_match.group(1).strip() if pfr_match else None

    # Extract Limbs
    limbs_pattern = r"Limbs:\s*([^\n]+?)(?=\n|$)"
    limbs_match = re.search(limbs_pattern, section_for_vitals)
    info["Limbs"] = limbs_match.group(1).strip() if limbs_match else None

    def extract_pupil_info(section_text):
        if not section_text:
            return "--"

        # Look for pupil section
        pupil_pattern = r"Pupil:\s*Right\s*(\d+)\s*mm,\s*([^,\n]+?)(?:,\s*[^,\n]+)?(?:\n|,\s*VA).*?Left\s*(\d+)\s*mm,\s*([^,\n]+?)(?:,\s*[^,\n]+)?(?:\n|,\s*IOP)"

        pupil_match = re.search(pupil_pattern, section_text, re.DOTALL)

        if not pupil_match:
            return "--"

        # Extract measurements and reactivity
        right_size, right_react, left_size, left_react = pupil_match.groups()

        # Format the output
        right_info = f"Right: {right_size}mm {right_react.strip()}"
        left_info = f"Left: {left_size}mm {left_react.strip()}"

        return f"{right_info} | {left_info}"

    # Add pupil information
    info["Pupil"] = extract_pupil_info(section_for_vitals)

    def extract_pregnancy_info(section_text, sex):
        # Return N/A for males
        if sex == "M":
            return "N/A"

        if not section_text:
            return "unspecified"

        # Check if pregnancy section exists by looking for key fields
        pregnancy_section_indicators = [
            r"LMP:",
            r"Pregnant:",
            r"Menopause:",
            r"Gravida:",
            r"Para:",
        ]

        # Check if any of the pregnancy-related fields exist in the text
        section_exists = any(
            re.search(pattern, section_text) for pattern in pregnancy_section_indicators
        )

        if not section_exists:
            return "unspecified"

        # Extract all relevant fields with more precise patterns
        # The (?:(?!\w+:).)*? pattern ensures we don't capture text that belongs to another field
        pregnancy_fields = {
            "LMP": re.search(r"LMP:\s*((?:(?!\w+:).)*?)(?:\s+\w+:|$)", section_text),
            "PMP": re.search(r"PMP:\s*((?:(?!\w+:).)*?)(?:\s+\w+:|$)", section_text),
            "Pregnant": re.search(
                r"Pregnant:\s*((?:(?!\w+:).)*?)(?:\s+\w+:|$)", section_text
            ),
            "Maturity": re.search(
                r"Maturity:\s*((?:(?!\w+:).)*?)(?:\s+\w+:|$)", section_text
            ),
            "Menopause": re.search(
                r"Menopause:\s*((?:(?!\w+:).)*?)(?:\s+\w+:|$)", section_text
            ),
            "EDC": re.search(r"EDC:\s*((?:(?!\w+:).)*?)(?:\s+\w+:|$|\n)", section_text),
            "Gravida": re.search(r"Gravida:\s*(\d*)", section_text),
            "Para": re.search(r"Para:\s*(\d*)", section_text),
            "AN clinic": re.search(
                r"AN clinic:\s*((?:(?!\w+:).)*?)(?:\s+\w+:|$|\n)", section_text
            ),
        }

        # Clean up extracted values
        values = {}
        for k, v in pregnancy_fields.items():
            if v:
                value = v.group(1).strip()
                # Only include non-empty values
                if value and value != "--":
                    values[k] = value

        # Format the output: field1: value1 | field2: value2 | ...
        if values:
            formatted_items = [f"{k}: {v}" for k, v in values.items()]
            return " | ".join(formatted_items)

        return "unspecified"

    # Add pregnancy information
    info["Pregnancy Status"] = extract_pregnancy_info(section_for_vitals, info["Sex"])

    # Extract H'stix and H'cue from Triage intervention section
    def extract_triage_intervention_readings(text):
        # Find the triage intervention section
        triage_section = re.search(
            r"Triage intervention(.*?)Consultation", text, re.DOTALL
        )

        if not triage_section:
            return None, None

        triage_text = triage_section.group(1)

        # Extract H'stix
        hstix_match = re.search(r"H'stix:\s*([\d.]+)\s*mmol/L", triage_text)
        hstix = f"{hstix_match.group(1)} mmol/L" if hstix_match else None

        # Extract H'cue
        hcue_match = re.search(r"H'cue:\s*([\d.]+)\s*g/dL", triage_text)
        hcue = f"{hcue_match.group(1)} g/dL" if hcue_match else None

        return hstix, hcue

    # Add H'stix and H'cue to info dictionary
    info["H'stix"], info["H'cue"] = extract_triage_intervention_readings(text)

    # Discharge Information
    specialty_match = re.search(r"Attending specialty:\s*(.+?)(?=\n|$)", text)
    info["Attending specialty"] = (
        specialty_match.group(1).strip() if specialty_match else None
    )

    trauma_match = re.search(r"Trauma type:\s*(.+?)(?=\n|$)", text)
    info["Trauma type"] = trauma_match.group(1).strip() if trauma_match else None

    dest_match = re.search(r"Discharge destination:\s*(.+?)(?=\n|$)", text)
    info["Discharge destination"] = dest_match.group(1).strip() if dest_match else None

    return info


def process_directory_medical_notes(directory_path):
    """
    Process all text files in a specific directory and return a DataFrame

    Args:
        directory_path (Path): Path object pointing to directory with text files
    Returns:
        pd.DataFrame: Combined DataFrame with all medical records from the directory
    """
    all_records = {}

    # Process only txt files that end with _anonymized.txt
    for txt_file in directory_path.glob("*_anonymized.txt"):
        try:
            print(
                f" ================== Processing file: {txt_file} ================== "
            )
            # Read and process the file
            with open(txt_file, "r", encoding="utf-8") as file:
                text = file.read()

            # Extract information
            info_dict = extract_medical_info(text)

            # Add file path information
            info_dict["Source_File"] = str(txt_file.name)
            info_dict["Source_Directory"] = str(
                txt_file.parent.relative_to(directory_path.parent)
            )

            # Use filename as index
            all_records[txt_file.name] = info_dict

            print(
                f" ================== Successfully processed: {txt_file} ================== "
            )

        except Exception as e:
            print(
                f" ================== Error processing {txt_file}: {str(e)} ================== "
            )

    if all_records:
        # Convert to DataFrame
        df = pd.DataFrame.from_dict(all_records, orient="index")
        # Sort index (filenames) for consistency
        df.sort_index(inplace=True)
        return df
    return None


def batch_process_medical_notes(input_dir, output_dir):
    """
    Process all anonymized text files in input directory and its subdirectories,
    creating separate CSV files for each subdirectory in the output directory.

    Args:
        input_dir (str): Path to directory containing anonymized text files
        output_dir (str): Path to directory where CSV files will be saved
    """
    # Convert to Path objects
    input_path = Path(input_dir)
    output_path = Path(output_dir)

    # Create output directory if it doesn't exist
    output_path.mkdir(parents=True, exist_ok=True)

    # Dictionary to store DataFrames by directory
    directory_dfs = {}

    # Process root directory
    root_df = process_directory_medical_notes(input_path)
    if root_df is not None:
        directory_dfs["root"] = root_df

    # Process each subdirectory
    for subdir in input_path.iterdir():
        if subdir.is_dir():
            subdir_df = process_directory_medical_notes(subdir)
            if subdir_df is not None:
                directory_dfs[subdir.name] = subdir_df

    # Save separate CSV files for each directory with proper integer handling
    total_records = 0
    for dir_name, df in directory_dfs.items():
        if not df.empty:
            # Convert float values to integers where appropriate
            # (Apply the conversion to all columns)
            for col in df.columns:
                if df[col].dtype == "float64":
                    df[col] = df[col].apply(
                        lambda x: int(x) if pd.notna(x) and float(x).is_integer() else x
                    )

            # Create filename
            if dir_name == "root":
                filename = "root.csv"
            else:
                filename = f"{dir_name}.csv"

            # Save to CSV without overriding data types
            output_file = output_path / filename
            df.to_csv(output_file, index=True, float_format="%.0f")

            total_records += len(df)
            print(f"Saved {len(df)} records to: {output_file}")

    print(f"\nProcessing complete. Total records processed: {total_records}")
    print(f"CSV files saved to: {output_dir}")

In [ ]:
batch_process_medical_notes(input_dir=r"..\data\past_case\txt", output_dir=r"..\data\past_case\csv")

#### 2.3 CSV to JSON Transformation
**Purpose**: Converts tabular clinical data into hierarchical JSON structure, optimizing data organization for machine learning applications and clinical decision support systems.


In [ ]:
import pandas as pd
import json
import os
from pathlib import Path
import glob
import sys

In [ ]:
def format_value(value):
    """
    Properly format values:
    - Convert integers without decimal points
    - Return original values for non-numeric types
    - Handle NaN values
    """
    if pd.isna(value):
        return ""

    # Convert float to int if it's an integer value
    if isinstance(value, float) and value.is_integer():
        return str(int(value))

    # Return string representation for any other value
    return str(value)


def convert_csv_to_json(csv_file_path, output_dir):
    """
    Convert a CSV file with medical triage data to JSON format.

    Args:
        csv_file_path (str): Path to the CSV file
        output_dir (str): Directory to save JSON files

    Returns:
        int: Number of records processed
    """
    # Read the CSV file with more control over data types
    df = pd.read_csv(csv_file_path)

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    records_processed = 0

    # Process each row in the dataframe
    for index, row in df.iterrows():
        # Create the JSON structure
        record = {
            "Demographics": {
                "Sex": format_value(row.get("Sex", "")),
                "Age": format_value(row.get("Age", "")),
                "Ambulatory status": format_value(row.get("Ambulatory status", "")),
                "Risk of fall": format_value(row.get("Risk of fall", "")),
                "Informant": format_value(row.get("Informant", "")),
                "Communication": format_value(row.get("Communication", "")),
                "Allergies": format_value(row.get("Allergy", "")),
                "ADR": format_value(row.get("ADR", "")),
                "Alerts": format_value(row.get("Alert", "")),
                "Past health": format_value(row.get("Past health", "")),
                "TOCC": format_value(row.get("TOCC", "")),
                "Referral (if any)": format_value(row.get("Referral", "")),
            },
            "Clinical Presentation": {
                "Chief complaint": format_value(row.get("Chief complaint", "")),
                "Condition on Arrival": format_value(
                    row.get("Condition on arrival", "")
                ),
            },
            "Vitals and Observations": {
                "Vital Signs": {
                    "GCS": format_value(row.get("GCS", "")),
                    "BP": format_value(row.get("BP", "")),
                    "PR/AR": format_value(row.get("PR/AR", "")),
                    "Temp": format_value(row.get("Temp", "")),
                    "SpO2": format_value(row.get("SpO2", "")),
                    "RR": format_value(row.get("RR", "")),
                    "PFR": format_value(row.get("PFR", "")),
                    "Limbs": format_value(row.get("Limbs", "")),
                    "Pupil": format_value(row.get("Pupil", "")),
                },
                "Pregnancy Status": format_value(row.get("Pregnancy Status", "")),
                "Triage Intervention": {
                    "H'stix": format_value(row.get("H'stix", "")),
                    "H'cue": format_value(row.get("H'cue", "")),
                },
            },
            "Case Disposition": {
                "Triage Category": format_value(row.get("Triage Category", "")),
                "Diagnosis": format_value(row.get("Diagnosis", "")),
                "Attending Specialty": format_value(row.get("Attending specialty", "")),
                "Trauma Type": format_value(row.get("Trauma type", "")),
                "Discharge Destination": format_value(
                    row.get("Discharge destination", "")
                ),
            },
        }

        # Get the source file name for output
        source_file = row.get("Source_File", f"case_{index}.txt")

        # Replace _formatted with _json in the filename
        # First, replace _anonymized.txt with _formatted.json (original format)
        temp_filename = source_file.replace("_anonymized.txt", "_formatted.json")
        # Then, replace _formatted with _json
        output_filename = temp_filename.replace("_formatted", "_json")

        # Create full output path
        output_path = os.path.join(output_dir, output_filename)

        # Save the JSON
        with open(output_path, "w", encoding="utf-8") as json_file:
            json.dump(record, json_file, indent=4, ensure_ascii=False)

        records_processed += 1

    return records_processed


def batch_process_directory(input_dir, output_dir):
    """
    Process all CSV files in input directory and subdirectories

    Args:
        input_dir (str): Path to directory containing CSV files
        output_dir (str): Path to directory where JSON files will be saved
    """
    input_path = Path(input_dir)
    output_path = Path(output_dir)

    # Ensure output directory exists
    os.makedirs(output_path, exist_ok=True)

    # Find all CSV files in the input directory
    csv_files = list(input_path.glob("*.csv"))

    if not csv_files:
        print(f"No CSV files found in {input_dir}")
        return

    total_files = len(csv_files)
    total_records = 0

    print(f"Found {total_files} CSV files to process.")

    # Process each CSV file
    for i, csv_file in enumerate(csv_files, 1):
        try:
            # Create subfolder named after CSV file (without extension)
            subfolder_name = csv_file.stem
            subfolder_path = output_path / subfolder_name

            print(
                f"[{i}/{total_files}] Processing {csv_file.name} => {subfolder_name}/"
            )

            # Convert CSV to JSON
            records = convert_csv_to_json(str(csv_file), str(subfolder_path))

            total_records += records
            print(f"  ✓ Processed {records} records from {csv_file.name}")

        except Exception as e:
            print(f"  ✗ Error processing {csv_file.name}: {str(e)}")

    print(f"\nProcessing complete!")
    print(f"Total CSV files processed: {total_files}")
    print(f"Total records converted: {total_records}")
    print(f"Output saved to: {output_dir}")


def post_process_json_files(output_base):
    """
    Post-process all formatted JSON files to:
    1. Remove 'Pregnancy Status' field if Sex is 'M'
    2. Rename 'Pregnancy Status' to 'Menstruation / Pregnancy Status' for non-male patients

    Args:
        output_base (str or Path): Path to the output directory containing formatted JSON files
    """
    # Convert output_base to a Path object if it's a string
    output_base = Path(output_base) if isinstance(output_base, str) else output_base

    print("\n" + "=" * 50)
    print("STARTING POST-PROCESSING")
    print("=" * 50)

    total_files = 0
    modified_files = 0

    # Process each category directory in the output folder
    for category_dir in output_base.glob("*"):
        if category_dir.is_dir():
            print(f"\nPost-processing directory: {category_dir.name}")

            # Process each formatted JSON file in the category directory
            for json_file in category_dir.glob("*.json"):
                total_files += 1

                try:
                    print(f"Post-processing: {json_file.name}")

                    # Load JSON data
                    with open(json_file, "r", encoding="utf-8") as f:
                        json_data = json.load(f)

                    modified = False

                    # Check if Demographics and Sex fields exist
                    if (
                        "Demographics" in json_data
                        and "Sex" in json_data["Demographics"]
                    ):
                        sex = json_data["Demographics"]["Sex"]

                        # Check if Vitals and Observations section exists
                        if "Vitals and Observations" in json_data:
                            vitals_section = json_data["Vitals and Observations"]

                            # Case 1: Remove Pregnancy Status for male patients
                            # Check for 'M' or any case variant of 'male'
                            if sex == "M" or sex.lower() == "male":
                                if "Pregnancy Status" in vitals_section:
                                    del vitals_section["Pregnancy Status"]
                                    modified = True
                                    print(
                                        f"  - Removed 'Pregnancy Status' field (patient is male)"
                                    )
                                if "Menstruation / Pregnancy Status" in vitals_section:
                                    del vitals_section[
                                        "Menstruation / Pregnancy Status"
                                    ]
                                    modified = True
                                    print(
                                        f"  - Removed 'Menstruation / Pregnancy Status' field (patient is male)"
                                    )
                            # Case 2: Rename Pregnancy Status for non-male patients
                            # Check if not 'M' and not any case variant of 'male'
                            elif (
                                sex != "M" and sex.lower() != "male"
                            ) and "Pregnancy Status" in vitals_section:
                                # Create new key with the old value
                                vitals_section["Menstruation / Pregnancy Status"] = (
                                    vitals_section["Pregnancy Status"]
                                )
                                # Delete the old key
                                del vitals_section["Pregnancy Status"]
                                modified = True
                                print(
                                    f"  - Renamed 'Pregnancy Status' to 'Menstruation / Pregnancy Status'"
                                )

                    # Save modified JSON data if changes were made
                    if modified:
                        with open(json_file, "w", encoding="utf-8") as f:
                            json.dump(json_data, f, indent=4)
                        modified_files += 1
                        print(f"  - Saved modifications to {json_file.name}")
                    else:
                        print(f"  - No modifications needed for {json_file.name}")

                except Exception as e:
                    print(f"Error post-processing file {json_file.name}: {str(e)}")
                    continue

    print(f"\nPost-processing complete. Files modified: {modified_files}/{total_files}")

In [ ]:
batch_process_directory(input_dir=r"..\data\past_case\csv", output_dir=r"..\data\past_case\json")

In [ ]:
post_process_json_files(output_base=r"..\data\past_case\json")

#### 2.4 Dataset Partitioning
**Purpose**: Implements stratified random sampling to create validation and test datasets while maintaining proportional representation across clinical categories.

In [ ]:
import os
from pathlib import Path
import random
import shutil
import math
import sys
import pandas as pd

In [ ]:
def scan_directory(root_directory, validation_files=None):
    """
    Scan the root directory and collect file paths by category.

    Args:
        root_directory (str): Path to directory containing JSON files
        validation_files (set, optional): Set of validation file paths to exclude

    Returns:
        tuple: Dictionary of category->files and total file count
    """
    category_files = {}
    total_files = 0

    try:
        for category in os.listdir(root_directory):
            category_path = os.path.join(root_directory, category)

            # Skip if not a directory
            if not os.path.isdir(category_path):
                continue

            # Get list of JSON files in this category
            json_files = []
            for root, _, files in os.walk(category_path):
                for file in files:
                    if file.endswith(".json"):
                        file_path = os.path.join(root, file)
                        # Only include if not in validation set (when validation_files is provided)
                        if (
                            validation_files is None
                            or file_path not in validation_files
                        ):
                            json_files.append(file_path)

            if json_files:
                category_files[category] = json_files
                total_files += len(json_files)

    except FileNotFoundError:
        print(f"Error: Directory not found: {root_directory}")
        sys.exit(1)
    except PermissionError:
        print(f"Error: Permission denied accessing: {root_directory}")
        sys.exit(1)

    return category_files, total_files


def calculate_proportions(category_files, total_files, num_files_to_select):
    """Calculate proportions and number of files to select from each category."""
    category_stats = {}
    files_to_select = {}

    for category, files in category_files.items():
        proportion = len(files) / total_files
        num_to_select = math.ceil(proportion * num_files_to_select)

        # Store statistics
        category_stats[category] = {
            "total_files": len(files),
            "proportion": proportion,
            "files_to_select": num_to_select,
        }

        # Randomly select files
        selected = random.sample(files, min(num_to_select, len(files)))
        files_to_select[category] = selected

    return category_stats, files_to_select


def adjust_selection(category_stats, files_to_select, num_files_to_select):
    """Adjust if we selected more than the target number of files due to rounding."""
    total_selected = sum(len(files) for files in files_to_select.values())

    if total_selected > num_files_to_select:
        # Sort categories by number of files selected
        sorted_categories = sorted(
            category_stats.items(), key=lambda x: x[1]["files_to_select"], reverse=True
        )

        # Remove files from largest categories until we reach our target
        for category, _ in sorted_categories:
            if total_selected <= num_files_to_select:
                break

            while (
                len(files_to_select[category]) > 1
                and total_selected > num_files_to_select
            ):
                files_to_select[category].pop()
                total_selected -= 1
                category_stats[category]["files_to_select"] -= 1

    return category_stats, files_to_select, total_selected


def move_files(files_to_select, output_directory):
    """Move selected files to the output directory."""
    moved_files = []

    for category, selected_files in files_to_select.items():
        # Create category subfolder in output directory
        category_output_dir = os.path.join(output_directory, category)
        os.makedirs(category_output_dir, exist_ok=True)

        # Move each selected file
        for file_path in selected_files:
            file_name = os.path.basename(file_path)
            output_path = os.path.join(category_output_dir, file_name)

            try:
                # Move file instead of copying
                shutil.move(file_path, output_path)
                moved_files.append(
                    {
                        "category": category,
                        "original_path": file_path,
                        "new_path": output_path,
                    }
                )
            except (IOError, PermissionError) as e:
                print(f"Error moving {file_path}: {str(e)}")

    return moved_files


def print_statistics(
    category_stats,
    total_files,
    total_selected,
    output_directory,
    moved_files,
    validation_files=None,
):
    """Print summary statistics and save results."""
    print(f"\nTotal files in dataset: {total_files}")
    if validation_files:
        print(f"Total files in validation set: {len(validation_files)}")
    print(f"Total files selected: {total_selected}")
    print("\nCategory statistics:")

    # Create DataFrame for statistics
    stats_data = []
    for cat, stats in category_stats.items():
        stats_data.append(
            {
                "Category": cat,
                "Total Files": stats["total_files"],
                "Proportion": f"{stats['proportion']:.2%}",
                "Files Selected": stats["files_to_select"],
                "Selection Proportion": f"{stats['files_to_select']/total_selected:.2%}",
            }
        )

    # Add totals row
    stats_data.append(
        {
            "Category": "TOTAL",
            "Total Files": total_files,
            "Proportion": "100.00%",
            "Files Selected": total_selected,
            "Selection Proportion": "100.00%",
        }
    )

    # Convert to DataFrame and print
    stats_df = pd.DataFrame(stats_data)
    print(stats_df.to_string(index=False))

    # Save list of moved files to CSV
    moved_files_df = pd.DataFrame(moved_files)

    if validation_files:
        csv_path = os.path.join(output_directory, "test_files.csv")
    else:
        csv_path = os.path.join(output_directory, "val_files.csv")

    moved_files_df.to_csv(csv_path, index=False)

    print(f"\nAll {total_selected} files have been moved to {output_directory}")
    print(f"List of moved files saved to {csv_path}")


def load_validation_files(validation_csv):
    """Load the list of validation files to exclude."""
    try:
        validation_df = pd.read_csv(validation_csv)
        # Get the original path for each validation file
        validation_files = set(validation_df["original_path"].tolist())
        print(f"Loaded {len(validation_files)} validation files to exclude")
        return validation_files
    except FileNotFoundError:
        print(f"Error: Validation CSV file not found: {validation_csv}")
        return set()
    except Exception as e:
        print(f"Error loading validation CSV: {str(e)}")
        return set()


def split_validation_set(json_dir, validation_dir, num_files=100):
    """Split dataset to create validation set."""
    print("\n=== STEP 4: Creating validation set ===")

    # Create output directory if it doesn't exist
    os.makedirs(validation_dir, exist_ok=True)

    # Step 1: Scan directory
    print("Scanning directory structure...")
    category_files, total_files = scan_directory(json_dir)

    if total_files == 0:
        print("Error: No JSON files found in the specified directory structure.")
        return False

    # Step 2: Calculate proportions
    print("Calculating proportions...")
    category_stats, files_to_select = calculate_proportions(
        category_files, total_files, num_files
    )

    # Step 3: Adjust selection if necessary
    category_stats, files_to_select, total_selected = adjust_selection(
        category_stats, files_to_select, num_files
    )

    # Step 4: Move files
    print("Moving files...")
    moved_files = move_files(files_to_select, validation_dir)

    # Step 5: Print statistics
    print_statistics(
        category_stats, total_files, total_selected, validation_dir, moved_files
    )

    return True


def split_test_set(json_dir, test_dir, validation_csv, num_files=100):
    """Split dataset to create test set, excluding validation files."""
    print("\n=== STEP 5: Creating test set ===")

    # Create output directory if it doesn't exist
    os.makedirs(test_dir, exist_ok=True)

    # Step 1: Load validation files to exclude
    validation_files = load_validation_files(validation_csv)
    if not validation_files:
        return False

    # Step 2: Scan directory, excluding validation files
    print("Scanning directory structure (excluding validation files)...")
    category_files, total_files = scan_directory(json_dir, validation_files)

    if total_files == 0:
        print("Error: No eligible JSON files found after excluding validation files.")
        return False

    # Step 3: Calculate proportions
    print("Calculating proportions...")
    category_stats, files_to_select = calculate_proportions(
        category_files, total_files, num_files
    )

    # Step 4: Adjust selection if necessary
    category_stats, files_to_select, total_selected = adjust_selection(
        category_stats, files_to_select, num_files
    )

    # Step 5: Move files
    print("Moving files...")
    moved_files = move_files(files_to_select, test_dir)

    # Step 6: Print statistics
    print_statistics(
        category_stats,
        total_files,
        total_selected,
        test_dir,
        moved_files,
        validation_files,
    )

    return True


### 2.5 Pipeline Execution and Data Source Configuration

#### Data Source Architecture
The preprocessing pipeline supports two distinct data processing workflows based on intended use:

**Database Building Workflow (`past_case`)**:
- **Purpose**: Processes historical medical cases for vector database construction
- **Output**: Complete structured dataset for vector database indexing
- **Next Stage**: Proceeds to `02_indexing.ipynb` for vector database population
- **Dataset Partitioning**: DISABLED - entire dataset preserved for comprehensive database

**Ground Truth Testing Workflow (`test_case`)**:
- **Purpose**: Processes evaluation cases for system validation and testing
- **Output**: Standardized format for direct RAG pipeline evaluation
- **Next Stage**: Direct input to MERC RAG pipeline for performance assessment
- **Dataset Partitioning**: DISABLED - maintains complete test case integrity

In [ ]:
# Configuration for historical case database building
base_dir = Path("../data/past_case")
# Process complete dataset for vector database construction
# Validation/test splitting: DISABLED
# Downstream processing: 02_indexing.ipynb

# Configuration for ground truth test case processing  
base_dir = Path("../data/test_case")
# Process for direct RAG evaluation
# Validation/test splitting: DISABLED
# Downstream processing: Direct to MERC RAG pipeline

input_dir = base_dir / "pdf"
txt_dir = base_dir / "txt"
csv_dir = base_dir / "csv"
json_dir = base_dir / "json"
validation_dir = base_dir / "val_case"
test_dir = base_dir / "test_case"

# Create output directories if they don't exist
for directory in [txt_dir, csv_dir, json_dir]:
    os.makedirs(directory, exist_ok=True)

# Step 1: PDF to TXT conversion
print("\n=== STEP 1: Converting PDFs to TXT ===")
batch_process_pdfs(str(input_dir), str(txt_dir))

# Step 2: TXT to CSV conversion
print("\n=== STEP 2: Extracting data from TXT to CSV ===")
batch_process_medical_notes(str(txt_dir), str(csv_dir))

# Pause for CSV review before proceeding to JSON conversion
print("\n=== CSV files have been generated ===")
print(f"CSV files are available for review at: {csv_dir}")
print(
    "Please review the CSV files now. You may delete any rows with poor quality data."
)

proceed = input("Type 'yes' to proceed with CSV to JSON conversion: ")

if proceed.lower() == "yes":
    # Step 3: CSV to JSON conversion
    print("\n=== STEP 3: Converting CSV to JSON ===")
    batch_process_directory(str(csv_dir), str(json_dir))
    post_process_json_files(str(json_dir))

    # Ask if user wants to create validation and test sets
    create_sets = input(
        "\nDo you want to create validation and test sets? (yes/no): "
    )

    if create_sets.lower() == "yes":
        # Calculate 10% of the total files for validation and test sets
        total_files = 0
        for category in os.listdir(str(json_dir)):
            category_path = os.path.join(str(json_dir), category)
            if os.path.isdir(category_path):
                for root, _, files in os.walk(category_path):
                    total_files += sum(
                        1 for file in files if file.endswith(".json")
                    )

        # Calculate 10% for validation and 10% for test sets
        val_size = int(total_files * 0.1)
        test_size = int(total_files * 0.1)

        print(f"\nTotal JSON files found: {total_files}")
        print(
            f"Will select approximately {val_size} files (10%) for validation set"
        )
        print(f"Will select approximately {test_size} files (10%) for test set")

        # Step 4: Create validation set
        validation_success = split_validation_set(
            str(json_dir), str(validation_dir), val_size
        )

        if validation_success:
            # Step 5: Create test set
            validation_csv = str(validation_dir / "val_files.csv")
            split_test_set(str(json_dir), str(test_dir), validation_csv, test_size)
else:
    print("\nCSV to JSON conversion cancelled. Pipeline paused.")
    print(
        "You can run the script again to complete the process after reviewing the CSV files."
    )

print("\nPipeline completed successfully!")